# Functions

Names are inspired by Brandon Sanderson’s books. Quantities are invented teaching examples; no knowledge of the books is required.

Main route: definitions → arguments → return values → scope → side effects → function interfaces.

## 1. Definition versus call

Running `def` creates a function. Its body runs when we call it.

In [ ]:
def greet_scadrial():
    print("Welcome to Scadrial, the world of Mistborn!")

print("Function defined; greeting not called yet.")
greet_scadrial()

## 2. Parameters and arguments

`crew_member` is a **parameter** in the definition. `"Kelsier"` and `"Vin"` are **arguments** supplied by the calls. Each call has its own local parameter bindings.

In [ ]:
def mistborn_greeting(crew_member):
    print(f"Hello, {crew_member}! Ready to burn some metals?")

mistborn_greeting("Kelsier")
mistborn_greeting("Vin")

## 3. Return values

The right-hand call produces a value that can be stored or used in another expression. `return` does not display the result.

In [ ]:
def stormlight_needed(spheres, stormlight_per_sphere):
    return spheres * stormlight_per_sphere

kaladin_stormlight = stormlight_needed(5, 2.5)
print(kaladin_stormlight)
print(stormlight_needed(2, 2.5) + 10)

In [ ]:
greet_vin = mistborn_greeting("Vin")
print("Stored result:", greet_vin)

**Explanation:** the function prints a message but has no explicit return value, so the call returns `None`. To reuse the text, return it:

In [ ]:
def mistborn_greeting_text(crew_member):
    return f"Hello, {crew_member}!"

greeting = mistborn_greeting_text("Vin")
print(greeting)
print(greeting + " Welcome to the crew.")

### A return ends the current call


In [ ]:
def has_enough_stormlight(reserve, required):
    if reserve >= required:
        return True
    return False

print(has_enough_stormlight(10, 3))
print(has_enough_stormlight(2, 3))

### Catch the missing case

What does the second call return? The function below is deliberately incomplete.

In [ ]:
def incomplete_stormlight_check(reserve, required):
    if reserve >= required:
        return True

print(incomplete_stormlight_check(10, 3))
print(incomplete_stormlight_check(2, 3))

It returns `None` when the condition is false. We can return the comparison directly so that both cases produce a Boolean value.

In [ ]:
def has_enough_stormlight(reserve, required):
    return reserve >= required

print(has_enough_stormlight(10, 3))
print(has_enough_stormlight(2, 3))

## 4. Local names and scope

**State** describes current information; **state space** contains possible states; **scope** describes where a name can be accessed directly.

The parameters and `remaining` below are local. Predict both calls. The second call does not inherit local values from the first.

In [ ]:
def remaining_stormlight(reserve, used):
    remaining = reserve - used
    return remaining

print(remaining_stormlight(10, 3))
print(remaining_stormlight(20, 4))
# print(remaining)  # NameError in a fresh kernel: this name is local.

### Reading a global name

Here “global” means defined at notebook/module level. Reading the name does not require `global`. Uppercase is a convention for a value intended to stay fixed.

In [ ]:
MAX_STORMLIGHT = 100

def free_capacity(reserve):
    return MAX_STORMLIGHT - reserve

print(free_capacity(30))

### Predict: local shadowing

Does assigning to the local name change the global name?

In [ ]:
stormlight_reserve = 10

def show_local_reserve():
    stormlight_reserve = 5
    print("Local:", stormlight_reserve)

show_local_reserve()
print("Global:", stormlight_reserve)

The values are 5 and 10. The local name shadows the global name inside the function. A returned object can remain usable even after the function has finished; local name visibility and object lifetime are different concepts.

### Rebinding a global name

`global` makes assignment target the global name. Rerunning the entire cell resets the reserve to 10 first.

In [ ]:
stormlight_reserve = 10

def use_stormlight_global():
    global stormlight_reserve
    stormlight_reserve = stormlight_reserve - 3

use_stormlight_global()
use_stormlight_global()
print(stormlight_reserve)

### Prefer an explicit input and result

The function calculates; the caller chooses whether to update its own name. Preconditions for this example: numeric inputs with `0 <= amount <= reserve`. They are documented assumptions, not automatic validation.

In [ ]:
def use_stormlight(reserve, amount):
    return reserve - amount

stormlight_reserve = 10
new_reserve = use_stormlight(stormlight_reserve, 3)
print("Original:", stormlight_reserve)
print("Calculated:", new_reserve)
stormlight_reserve = new_reserve
print("Updated:", stormlight_reserve)

## 5. Side effects and pure functions

A side effect is observable beyond the returned value, such as output or modification of externally accessible data. Local intermediate calculations are not side effects by themselves.

For the numeric inputs used here, `use_stormlight` is pure: the same arguments give the same result, and the function has no side effects. The next version also prints.

In [ ]:
def use_stormlight_with_report(reserve, amount):
    remaining = reserve - amount
    print(f"Stormlight remaining: {remaining}")
    return remaining

result = use_stormlight_with_report(10, 3)
print("Returned:", result)

### Reading changing external state

A function can have no side effects yet still be impure: its result may depend on changing external state. Predict both results for the same argument.

In [ ]:
stormlight_bonus = 2

def boosted_stormlight(reserve):
    return reserve + stormlight_bonus

print(boosted_stormlight(10))
stormlight_bonus = 5
print(boosted_stormlight(10))

### Separate calculation and presentation

Side effects are useful. Keeping them explicit makes the calculation easier to reuse and check.

In [ ]:
remaining = use_stormlight(10, 3)
print(f"Kaladin has {remaining} units left.")

## 6. Positional and keyword arguments

Positional arguments match by order; keyword arguments match by parameter name. In these simple calls, positional arguments must come before keyword arguments.

In [ ]:
def metal_reserves(character, pewter, steel):
    return f"{character} has {pewter} pewter and {steel} steel in reserve."

print(metal_reserves("Vin", 3, 5))
print(metal_reserves(steel=5, character="Vin", pewter=3))
print(metal_reserves("Kelsier", pewter=4, steel=7))

## 7. Default parameter values

If the argument is omitted, the default is used. For these ordinary parameters, required parameters come before parameters with defaults.

In [ ]:
def shardblade_summon(name="Syl"):
    return f"{name} is summoned!"

print(shardblade_summon())
print(shardblade_summon("Pattern"))

## 8. Type hints

Annotations document expected types and support editor/static checking tools. Python does not enforce them automatically at runtime. They do not automatically convert an argument to the annotated type.

In [ ]:
def stormlight_needed(spheres: int, stormlight_per_sphere: float) -> float:
    return spheres * stormlight_per_sphere

print(stormlight_needed(5, 2.5))

## 9. Returning several pieces of information

The function returns one **tuple** containing three elements. A tuple is an immutable sequence. Unpacking assigns the elements to separate names; the number of names must match the number of elements here.

In [ ]:
def allomancer_info(name: str, metal: str) -> tuple:
    return name, metal, f"{name} is burning {metal}!"

info = allomancer_info("Vin", "pewter")
print(info)
name, metal, message = info
print(name)
print(message)

## 10. Docstrings

A docstring is the string at the start of a function body. Explain the purpose, inputs, result and any important assumptions. `help()` displays the documentation.

In [ ]:
def spren_bond(surgebinder: str, spren: str) -> str:
    """Return a sentence describing a bond.

    surgebinder and spren are the two names to include.
    This function returns text; it does not print it.
    """
    return f"{surgebinder} has bonded with {spren}."

help(spren_bond)
bond = spren_bond("Kaladin", "Syl")
print(bond)

## Extra: changing a list versus rebinding a name

The parameter receives a reference to the same list object. Appending modifies that object, so the caller sees the change. No `global` is needed. This is a side effect.

In [ ]:
def add_crew_member(crew, name):
    crew.append(name)

crew = ["Vin", "Kelsier"]
add_crew_member(crew, "Sazed")
print(crew)

Assigning a new list to the local parameter only rebinds that local name. It does not rebind the caller’s name.

In [ ]:
def replace_local_crew(crew):
    crew = ["Marsh"]
    return crew

replacement = replace_local_crew(crew)
print("Original:", crew)
print("Returned:", replacement)

## A mutable default

Default values are evaluated when the definition runs. A default list can therefore be shared between calls. Predict both printed results.

In [ ]:
def remember_member(name, crew=[]):
    crew.append(name)
    return crew

print(remember_member("Vin"))
print(remember_member("Kelsier"))

When each omitted argument should mean a fresh list, use `None` and create the list inside the function.

In [ ]:
def remember_member_fresh(name, crew=None):
    if crew is None:
        crew = []
    crew.append(name)
    return crew

print(remember_member_fresh("Vin"))
print(remember_member_fresh("Kelsier"))